# DefectForge M11 — SDXL inpainting LoRA

Thin Colab wrapper around `src/training/train_inpaint_lora.py`. It does not duplicate the training loop. Run only on an **L4 (24 GB)** runtime.

## 1. Runtime and GPU preflight

Choose **Runtime → Change runtime type → L4 GPU**. The cell fails before installation or model download when CUDA is absent or total VRAM is below 20 GiB.

In [ ]:
import subprocess

import torch

assert torch.cuda.is_available(), "Select an L4 GPU runtime first"
props = torch.cuda.get_device_properties(0)
total_gib = props.total_memory / 2**30
print(props.name, f"{total_gib:.1f} GiB")
assert total_gib >= 20, "M11 requires an L4-class 24 GB GPU"
subprocess.run(["nvidia-smi"], check=True)

## 2. Mount Drive and stage data locally

Before running, place `defectforge_source.zip` and `m11_sdxl_inputs.zip` in `MyDrive/sdg-portfolio/01-defectforge-visa/`. Both archives are copied and extracted under `/content`; training never reads images from mounted Drive.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/sdg-portfolio/01-defectforge-visa')
PROJECT_ROOT = Path('/content/defectforge')
DATA_ROOT = Path('/content/data/01-defectforge')
for archive in ('defectforge_source.zip', 'm11_sdxl_inputs.zip'):
    assert (DRIVE_ROOT / archive).is_file(), f'Missing {archive} in Drive'
shutil.copy2(DRIVE_ROOT / 'defectforge_source.zip', '/content/defectforge_source.zip')
shutil.copy2(DRIVE_ROOT / 'm11_sdxl_inputs.zip', '/content/m11_sdxl_inputs.zip')
subprocess.run(['unzip', '-q', '/content/defectforge_source.zip', '-d', '/content'], check=True)
subprocess.run(['unzip', '-q', '/content/m11_sdxl_inputs.zip', '-d', '/content/data'], check=True)
assert (PROJECT_ROOT / 'pyproject.toml').is_file()
assert (DATA_ROOT / 'raw/VisA').is_dir()

## 3. Secrets and reproducible environment

Create a Colab Secret named `HF_TOKEN` and grant this notebook access. No credential is stored in the notebook or output files.

In [ ]:
import os
import subprocess

import yaml
from google.colab import userdata

token = userdata.get('HF_TOKEN')
assert token, 'Grant this notebook access to the HF_TOKEN Colab Secret'
os.environ['HF_TOKEN'] = token
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=PROJECT_ROOT, check=True)
paths = yaml.safe_load((PROJECT_ROOT / 'configs/paths.yaml').read_text())
paths['data_root'] = str(DATA_ROOT)
paths['dotenv'] = '/content/defectforge-no-secrets.env'
COLAB_PATHS = PROJECT_ROOT / 'configs/paths_colab.yaml'
COLAB_PATHS.write_text(yaml.safe_dump(paths, sort_keys=False), encoding='utf-8')

## 4. Dry-run, training, and automatic resume

Each object has a unique Drive checkpoint directory. If a complete checkpoint exists, the same command adds `--resume-from-checkpoint latest`; otherwise it starts fresh. Expected formal budget: 400 steps per object. Record actual wall time and compute-unit change after both runs.

In [ ]:
import time

CONFIG = PROJECT_ROOT / 'configs/lora_sdxl.yaml'
LOCAL_RUNS = DATA_ROOT / 'runs/lora_sdxl'
DRIVE_RUNS = DRIVE_ROOT / 'runs/lora_sdxl'
for object_name in ('pcb1', 'capsules'):
    subprocess.run(['uv', 'run', '--frozen', 'python', 'src/training/train_inpaint_lora.py', '--paths', str(COLAB_PATHS), '--config', str(CONFIG), '--object', object_name, '--dry-run'], cwd=PROJECT_ROOT, check=True)
    local_output = LOCAL_RUNS / object_name / 'seed_42'
    drive_output = DRIVE_RUNS / object_name / 'seed_42'
    local_output.parent.mkdir(parents=True, exist_ok=True)
    drive_output.mkdir(parents=True, exist_ok=True)
    checkpoints = sorted(drive_output.glob('checkpoint-*'))
    if checkpoints and not local_output.exists():
        shutil.copytree(drive_output, local_output)
    command = ['uv', 'run', '--frozen', 'python', 'src/training/train_inpaint_lora.py', '--paths', str(COLAB_PATHS), '--config', str(CONFIG), '--object', object_name, '--output-dir', str(local_output), '--drive-sync', str(drive_output)]
    if checkpoints:
        command += ['--resume-from-checkpoint', 'latest']
    started = time.perf_counter()
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print(object_name, f'{time.perf_counter() - started:.1f}s')

## 5. Validate and collect results

The validator independently checks both objects, frozen checksums, checkpoint inventory, adapters, held-out samples, blocklist, and fresh PEFT reload. Download the validation JSON plus each object's final adapters, training report, and sample panels into `results/colab/lora_sdxl/` locally.

In [ ]:
RESULT_ROOT = DRIVE_ROOT / 'results/lora_sdxl'
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
validation = RESULT_ROOT / 'lora_sdxl_validation.json'
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/validate_lora_run.py', '--paths', str(COLAB_PATHS), '--config', str(CONFIG), '--run-root', str(LOCAL_RUNS), '--reload', '--output', str(validation)], cwd=PROJECT_ROOT, check=True)
for object_name in ('pcb1', 'capsules'):
    source = LOCAL_RUNS / object_name / 'seed_42'
    target = RESULT_ROOT / object_name
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source / 'training_report.json', target / 'training_report.json')
    shutil.copytree(source / 'final', target / 'final', dirs_exist_ok=True)
    shutil.copytree(source / 'samples', target / 'samples', dirs_exist_ok=True)
print('Validated results:', RESULT_ROOT)